
# Skill anatomy and the SKILL.md format

An **agent skill** is a folder of instructions and resources that an agent loads *on demand* to become better at a recurring task. The whole skill is anchored by a single file - `SKILL.md` - which is really two documents stitched together:

1. **YAML frontmatter** - machine-readable metadata used for *discovery and routing*.
2. **A Markdown body** - the natural-language *instructions* the agent follows once the skill is activated.

This split is what makes skills efficient. At routing time the agent only needs the tiny frontmatter to decide *whether* a skill is relevant; the full body is loaded only after the skill is chosen; bundled reference files are loaded later still, and only if the workflow actually cites them. This staged loading is called **progressive disclosure**, and the anatomy of `SKILL.md` exists to support it.

> **A note on what is actually required.** In the portable Agent Skills standard, the only strictly required frontmatter fields are **`name`** and **`description`**. `allowed-tools` and `license` are real, commonly used optional fields. `version`, `tags`, `author`, and `dependencies` are organizational extensions that teams and registries layer on top - useful, but not part of the minimal spec.

We use two real, on-disk skills that ship next to this notebook under `example-skills/` - never inlined here, exactly as skills are organized in production.

In [1]:
import os
import json
from dataclasses import dataclass, field
from pathlib import Path

from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import create_react_agent

import yaml  # PyYAML — reads the YAML frontmatter block

/usr/local/python/3.12.1/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Initializing the LLMs

In [2]:
# A small, fast model is plenty for these illustrative examples
llm = ChatOpenAI(model="gpt-4o-mini", api_key=os.getenv("OPENAI_API_KEY", "").strip(), temperature=0)
print("Model ready:", llm.model)

# The two example skills bundled with this notebook. They are real directories on disk, not strings defined in a cell.
EXAMPLE_SKILLS = Path("example-skills")
CHANGELOG_SKILL = EXAMPLE_SKILLS / "changelog-generator"
GLOSSARY_SKILL = EXAMPLE_SKILLS / "glossary-lookup"

print("changelog-generator skill is a folder, not a single file:")
for p in sorted(CHANGELOG_SKILL.rglob("*")):
    if p.is_file():
        print("  ", p.relative_to(EXAMPLE_SKILLS))

Model ready: gpt-4o-mini
changelog-generator skill is a folder, not a single file:
   changelog-generator/.ipynb_checkpoints/SKILL-checkpoint.md
   changelog-generator/SKILL.md
   changelog-generator/assets/.ipynb_checkpoints/changelog-template-checkpoint.md
   changelog-generator/assets/changelog-template.md
   changelog-generator/references/.ipynb_checkpoints/keep-a-changelog-checkpoint.md
   changelog-generator/references/keep-a-changelog.md
   changelog-generator/scripts/.ipynb_checkpoints/parse_git_log-checkpoint.py
   changelog-generator/scripts/parse_git_log.py


The directory listing already tells the anatomy story: one `SKILL.md` at the root, plus optional `references/`, `scripts/`, and `assets/` subfolders. Only `SKILL.md` is required; the subfolders hold resources that are pulled in lazily. We will dissect `SKILL.md` first, then return to the resources in later notebooks of this section.


## 1. Parsing a real SKILL.md
The frontmatter is fenced by `---` lines at the very top of the file. Splitting on the first two fences cleanly separates the YAML metadata from the Markdown body.

In [3]:
def parse_skill_md(skill_md_path):
    """Split a SKILL.md file into its YAML frontmatter and Markdown body.

    A SKILL.md is two documents in one file: a YAML metadata block fenced by `---` lines, followed by the Markdown instruction body.

    Returns:
        (frontmatter: dict, body: str)
    """
    text = Path(skill_md_path).read_text(encoding="utf-8")
    if not text.startswith("---"):
        raise ValueError("SKILL.md must start with a '---' frontmatter block")
    # text.split('---', 2) -> ['', '<yaml>', '<body>']
    _, frontmatter_yaml, body = text.split("---", 2)
    return yaml.safe_load(frontmatter_yaml), body.strip()


frontmatter, body = parse_skill_md(CHANGELOG_SKILL / "SKILL.md")

print("Frontmatter fields:", list(frontmatter))
print("\nname:       ", frontmatter["name"])
print("description:", frontmatter["description"])
print("\nBody begins:\n")
print("\n".join(body.splitlines()[:6]))

Frontmatter fields: ['name', 'description', 'version', 'allowed-tools', 'tags', 'author', 'license']

name:        changelog-generator
description: Generates a Keep a Changelog formatted CHANGELOG entry from a release's git commit history, grouping changes into Added, Changed, Fixed, and Removed

Body begins:

# Changelog Generator

Turns the raw commit history of a release into a clean, human-readable
CHANGELOG entry that follows the Keep a Changelog convention. The goal is a
changelog written for humans reading the release notes, not a dump of git log.



`parse_skill_md` is the single most reused helper when working with skills - routing, validation, and activation all start by separating these two halves. Notice that `yaml.safe_load` turns the frontmatter into an ordinary Python dict (so `allowed-tools` becomes a list, `version` a string), while the body stays as raw Markdown text that an LLM will read verbatim. We will reuse this function throughout the notebook.

## 2. Progressive disclosure in token terms
The reason for the two-part format is cost. At startup the agent preloads only the **`name` and `description`** of every available skill - that is all it needs to decide *whether* a skill is relevant. It loads the **body** only for the skill it actually selects, and loads **reference files** only when a workflow step cites them. Let's measure those three tiers for the changelog skill.

In [4]:
def estimate_tokens(text):
    """Estimate token count: tiktoken when installed, else ~4 chars/token."""
    try:
        import tiktoken
        return len(tiktoken.get_encoding("cl100k_base").encode(text))
    except Exception:
        return len(text) // 4


# Tier 1 is only what the router needs to compare skills: the name and description.
# This is the metadata that stays resident in context for *every* installed skill, so it is the line item that must stay tiny.
routing_metadata = f"{frontmatter['name']}: {frontmatter['description']}"
reference = (CHANGELOG_SKILL / "references" / "keep-a-changelog.md").read_text()

print(f"Tier 1  routing    (name + description)    ~{estimate_tokens(routing_metadata):>4} tokens")
print(f"Tier 2  activation (full instruction body) ~{estimate_tokens(body):>4} tokens")
print(f"Tier 3  on demand  (one reference file)    ~{estimate_tokens(reference):>4} tokens")

Tier 1  routing    (name + description)    ~  33 tokens
Tier 2  activation (full instruction body) ~ 843 tokens
Tier 3  on demand  (one reference file)    ~ 494 tokens


Tier 1 is what the router compares across *every* installed skill, so keeping it tiny matters. Tier 2 only enters the context window once this skill is chosen. Tier 3 is paid for only if the agent actually reaches the workflow step that references `keep-a-changelog.md`. A skill with ten heavy reference files still costs only its `name` and `description` at routing time - that is the entire point of the folder layout.

## 3. The frontmatter fields - and which are actually required
Here are the changelog skill's frontmatter fields, each tagged with how essential it really is. This is where the gap between *the standard* and *common house conventions* becomes concrete.

In [5]:
# Only `name` + `description` are required by the portable standard. 
# The rest are real-but-optional, or organizational extensions layered on by teams/registries.
FIELD_ROLE = {
    "name":          "required (spec)   unique identifier, kebab-case",
    "description":   "required (spec)   the routing signal",
    "allowed-tools": "common optional   tool allowlist (e.g. Claude Code)",
    "license":       "common optional   distribution license (SPDX id)",
    "version":       "org extension     semantic version for change tracking",
    "tags":          "org extension     catalog / discovery facets",
    "author":        "org extension     ownership & approval workflows",
    "dependencies":  "org extension     other skills required first",
}

for key, value in frontmatter.items():
    shown = ", ".join(value) if isinstance(value, list) else value
    print(f"{key:<14} {FIELD_ROLE.get(key, 'unknown')}")
    print(f"{'':<14} = {shown}\n")

name           required (spec)   unique identifier, kebab-case
               = changelog-generator

description    required (spec)   the routing signal
               = Generates a Keep a Changelog formatted CHANGELOG entry from a release's git commit history, grouping changes into Added, Changed, Fixed, and Removed

version        org extension     semantic version for change tracking
               = 1.1.0

allowed-tools  common optional   tool allowlist (e.g. Claude Code)
               = read_file, list_directory, run_git_log, write_file

tags           org extension     catalog / discovery facets
               = engineering, documentation, release-management

author         org extension     ownership & approval workflows
               = developer-experience-team

license        common optional   distribution license (SPDX id)
               = MIT



The takeaway: a perfectly valid skill can carry just `name` and `description`. The `description` is doing the heavy lifting - it is the text the router embeds and matches against the user's request, so it should read like the task a user would ask for. `allowed-tools` is the other field that changes runtime behavior (it scopes the toolset, demonstrated in section 7). Everything else here - `version`, `tags`, `author` - is governance metadata: valuable for managing a catalog of skills, but invisible to the agent's reasoning.

## 4. The instruction body's sections
The body is plain Markdown, but a conventional set of `##` sections gives the agent a predictable structure: when to use the skill, what it needs, the workflow, decision rules, constraints, and output format. Let's extract them.

In [6]:
def split_sections(body):
    """Return an ordered dict of H2 section title -> section text."""
    sections, current = {}, None
    for line in body.splitlines():
        if line.startswith("## "):
            current = line[3:].strip()
            sections[current] = []
        elif current is not None:
            sections[current].append(line)
    return {title: "\n".join(lines).strip() for title, lines in sections.items()}


sections = split_sections(body)
print("Body sections, in order:")
for i, title in enumerate(sections, 1):
    print(f"  {i}. {title}")

print("\n--- 'When to Use This Skill' ---")
print(sections["When to Use This Skill"])

Body sections, in order:
  1. When to Use This Skill
  2. Required Context
  3. Workflow
  4. Decision Rules
  5. Constraints
  6. Output Format
  7. Resources

--- 'When to Use This Skill' ---
**Activate when:**
- The user asks to "write a changelog", "draft release notes", or "summarize commits for a release"
- A release or tag is being prepared and the commits since the last tag need to be summarized
- The user provides a commit range and wants it grouped by change type

**Do NOT activate when:**
- The user wants a single conventional commit message for staged changes (use `git-commit-message`)
- The user wants a prose blog-style announcement rather than a structured changelog
- There is no commit history available yet (ask the user to provide a commit range first)


Each section has a job. **When to Use** prevents the skill from firing on the wrong request (note its explicit *Do NOT activate* list). **Required Context** is a pre-flight checklist. **Workflow** is the ordered procedure and is where reference files and scripts get cited at their point of use. **Constraints** are guardrails, and **Output Format** pins down the deliverable. The **Instruction body** notebook in this section is devoted to writing these well; here we just confirm the structure is machine-readable.

## 5. Building a SKILL.md programmatically
Authoring skills by hand is normal, but generating them is useful for scaffolding, migrations, and tests. A small typed builder keeps required fields explicit and emits optional fields only when set - so minimal skills stay minimal.

In [7]:
@dataclass
class SkillFrontmatter:
    """Typed builder for the SKILL.md frontmatter block.

    `name` and `description` are required by the standard. `version` defaults to "1.0.0" and is always emitted because this repo's convention requires it.
    `allowed-tools` and `tags` are optional and appear only when set, so minimal skills stay minimal.
    """
    name: str
    description: str
    version: str = "1.0.0"
    allowed_tools: list = field(default_factory=list)
    tags: list = field(default_factory=list)

    def to_yaml(self):
        # name + description (spec-required) and version (house convention) always go out.
        data = {"name": self.name, "description": self.description, "version": self.version}
        # Optional fields are written only when non-empty, keeping the block lean.
        if self.allowed_tools:
            data["allowed-tools"] = self.allowed_tools
        if self.tags:
            data["tags"] = self.tags
        return yaml.safe_dump(data, sort_keys=False).strip()


def build_skill_md(fm, title, body_md):
    """Assemble a complete SKILL.md string from frontmatter + title + body."""
    return f"---\n{fm.to_yaml()}\n---\n\n# {title}\n\n{body_md.strip()}\n"


fm = SkillFrontmatter(
    name="pr-summary",
    description="Summarizes an open pull request into a short reviewer-facing digest of intent, risk, and test coverage",
    allowed_tools=["read_file", "list_directory"],
    tags=["engineering", "code-review"],
)

skill_md = build_skill_md(
    fm,
    title="PR Summary",
    body_md=(
        "## When to Use\n"
        "When asked to summarize or triage an open pull request for reviewers.\n\n"
        "## Workflow\n"
        "1. Read the PR diff with `read_file`.\n"
        "2. Identify intent, risky changes, and whether tests were added.\n"
        "3. Produce a 3-bullet digest: what changed, the risk, and test coverage."
    ),
)
print(skill_md)

---
name: pr-summary
description: Summarizes an open pull request into a short reviewer-facing digest of
  intent, risk, and test coverage
version: 1.0.0
allowed-tools:
- read_file
- list_directory
tags:
- engineering
- code-review
---

# PR Summary

## When to Use
When asked to summarize or triage an open pull request for reviewers.

## Workflow
1. Read the PR diff with `read_file`.
2. Identify intent, risky changes, and whether tests were added.
3. Produce a 3-bullet digest: what changed, the risk, and test coverage.




The builder produces exactly the same two-part shape we parsed in section 1, which
means it round-trips: a generated skill must parse back into a frontmatter dict and a
body, and must carry the two required fields. That round-trip is the smallest useful
"is this a real skill?" check.

In [8]:
import tempfile

with tempfile.TemporaryDirectory() as tmp:
    path = Path(tmp) / "SKILL.md"
    path.write_text(skill_md, encoding="utf-8")
    fm_back, body_back = parse_skill_md(path)

assert fm_back["name"] == "pr-summary", "name did not survive the round-trip"
assert fm_back["description"], "description is required and must be non-empty"
print("Round-trip OK:", fm_back["name"], "| body ~", estimate_tokens(body_back), "tokens")

Round-trip OK: pr-summary | body ~ 71 tokens


## 6. The minimum viable skill

Not every skill needs phases, decision rules, and bundled resources. The `glossary-lookup` skill bundled here is a single `SKILL.md` carrying *only* the two fields the standard requires - `name` and `description` - plus two body sections, and it is a perfectly valid skill. It deliberately omits even `version` to show the true spec floor.

In [9]:
glossary_fm, glossary_body = parse_skill_md(GLOSSARY_SKILL / "SKILL.md")

print("Minimal frontmatter fields:", list(glossary_fm))
print("Body sections:", list(split_sections(glossary_body)))
print(f"Entire body is ~{estimate_tokens(glossary_body)} tokens")

Minimal frontmatter fields: ['name', 'description']
Body sections: ['When to Use', 'Workflow']
Entire body is ~120 tokens


Two fields and two sections is genuinely all the standard demands. A house validator might still *warn* that `version` is missing - but a warning is not an error, and the skill routes and runs exactly as well as a fully-decorated one. Reach for a minimal skill when the task is a short, well-understood procedure with no external reference material. Reach for the full structure (like `changelog-generator`) when there are real decision points, guardrails, a specified output, or resources worth loading lazily. The format scales down as gracefully as it scales up.

## 7. Loading a skill into a LangGraph agent
This is where anatomy becomes behavior. A `SKILL.md` turns into a working agent in two moves:
1. the Markdown **body becomes the system prompt** - the agent's instructions, and
2. **`allowed-tools` filters the toolset** the agent is allowed to call.

Below we wire the *changelog-generator* skill to a LangGraph ReAct agent. We define a small set of tools, scope them with the skill's `allowed-tools`, and pass the body as the prompt. A tiny in-memory "repo" stands in for a real one so the tools are runnable.

In [11]:
# A minimal fake project the tools operate on, so the demo actually runs.
FAKE_REPO = {
    "CHANGELOG.md": "# Changelog\n",
    "commits": [
        "feat(search): add pagination to results",
        "fix(api): handle empty query without crashing",
        "docs: update README",
        "feat!: drop support for legacy tokens",
    ],
}


@tool
def list_directory() -> str:
    """List the files in the repository."""
    return "\n".join(k for k in FAKE_REPO if k != "commits")


@tool
def read_file(path: str) -> str:
    """Read a file from the repository."""
    return FAKE_REPO.get(path, f"(no such file: {path})")


@tool
def run_git_log() -> str:
    """Return commit subjects for the release range, newest first."""
    return "\n".join(FAKE_REPO["commits"])


@tool
def write_file(path: str, content: str) -> str:
    """Write content to a file in the repository."""
    FAKE_REPO[path] = content
    return f"wrote {len(content)} chars to {path}"


ALL_TOOLS = {t.name: t for t in [list_directory, read_file, run_git_log, write_file]}

# Scope the toolset to exactly what the skill's frontmatter allows. A tool the skill does not list is simply never given to the agent.
allowed = frontmatter.get("allowed-tools", [])
scoped_tools = [ALL_TOOLS[name] for name in allowed if name in ALL_TOOLS]
print("Skill allows:", allowed)
print("Tools wired to the agent:", [t.name for t in scoped_tools])

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# The skill body IS the system prompt — the agent now behaves as this skill.
agent = create_react_agent(llm, scoped_tools, prompt=body)

result = agent.invoke({
    "messages": [("user", "Draft the changelog for version 1.4.0 (released 2026-06-27).")]
})
print("\n--- Agent output ---")
print(result["messages"][-1].content)

Skill allows: ['read_file', 'list_directory', 'run_git_log', 'write_file']
Tools wired to the agent: ['read_file', 'list_directory', 'run_git_log', 'write_file']


/tmp/ipykernel_726/2969852370.py:49: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(llm, scoped_tools, prompt=body)



--- Agent output ---
Please provide the commit range or the previous tag to diff against for version 1.4.0.


Nothing about the agent is changelog-specific in code - we never wrote changelog logic. The behavior came entirely from the `SKILL.md`: the body told the agent to collect commits, categorize them by the Keep a Changelog mapping, and render the entry, while `allowed-tools` decided it could use `run_git_log` and `write_file` but, say, not a network tool. Swap in a different `SKILL.md` and the same harness becomes a different specialist. That is the practical payoff of the anatomy.

- A skill is a folder whose required entry point is **`SKILL.md`** - YAML frontmatter for routing plus a Markdown body for instructions.
- The two-part format exists to enable progressive disclosure: cheap frontmatter at routing time, the body on activation, reference files only when cited.
- `name` and `description` are the only required fields. `allowed-tools` and `license` are common optionals; `version`, `tags`, `author`, and `dependencies` are   organizational extensions. The `description` is the routing signal — invest in it.
- The body's conventional sections (When to Use, Required Context, Workflow, Decision Rules, Constraints, Output Format) make a skill predictable and machine-readable.
- A skill becomes an agent by using the body as the system prompt and `allowed-tools` as the tool scope - shown here with `create_react_agent`.